# Exercises XP – Week 5, Day 1: Diabetes Classification
Goal: predict whether a person has diabetes with **logistic regression**.
Dataset: `diabetes_prediction_dataset.csv` (100,000 people).

## 🌟 Exercise 1: Understanding the Problem and Data Collection

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score)

LOCAL_FILE = "diabetes_prediction_dataset.csv"
GITHUB_URL = "https://raw.githubusercontent.com/jenny-palti/DI-Bootcamp2026/main/Week5/Day1/diabetes_prediction_dataset.csv"
df = pd.read_csv(LOCAL_FILE if os.path.exists(LOCAL_FILE) else GITHUB_URL)
print(df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.describe().round(2)

In [ ]:
counts = df["diabetes"].value_counts()
print("Negative cases (no diabetes):", counts[0])
print("Positive cases (diabetes):   ", counts[1])
print(f"Share of positive cases: {counts[1] / len(df):.1%}")

sns.countplot(data=df, x="diabetes", hue="diabetes", palette="Set2", legend=False)
plt.title("Diabetes cases (0 = no, 1 = yes)")
plt.show()

The data is **imbalanced**: only about **8.5%** of people have diabetes. We'll keep this in mind when evaluating (accuracy alone could be misleading).

**Preparing the features:** `gender` and `smoking_history` are text, so we one-hot encode them. We remove duplicate rows so the same person doesn't end up in both train and test.

In [ ]:
df = df.drop_duplicates()
df = df[df["gender"] != "Other"]          # only 18 people – too few to learn from
X = pd.get_dummies(df.drop(columns="diabetes"), columns=["gender", "smoking_history"], drop_first=True, dtype=int)
y = df["diabetes"]

# Stratified split: keeps the same share of positive cases in train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Positive share – train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

## 🌟 Exercise 2: Model Picking and Standardization
**Which model?** This is a **binary classification** problem (diabetes yes/no), so we can use classification models such as **logistic regression**, decision trees, random forests, KNN or SVM. **Logistic regression** is a good first choice: it's simple, fast on 100,000 rows, gives a **probability** of diabetes (useful for setting a risk threshold), and its coefficients show how each factor (glucose, HbA1c, age…) affects the risk – which matters in a medical context.

**Do we need to standardize?** **Yes.** The features have very different scales (age up to 80, blood glucose up to 300, the 0/1 columns). Logistic regression is trained with gradient-based optimisation and regularisation, which both work better and treat the features fairly when they're on a similar scale. We fit the `StandardScaler` on the **training data only** and then apply it to the test data, so no information from the test set leaks into training.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Mean after scaling (≈0):", X_train_s.mean(axis=0).round(2)[:5])
print("Std after scaling (≈1): ", X_train_s.std(axis=0).round(2)[:5])

## 🌟 Exercise 3: Model Training

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

coefficients = pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print("Coefficients (on standardized features):")
coefficients.round(3)

## 🌟 Exercise 4: Evaluation Metrics

In [ ]:
y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
baseline = 1 - y_test.mean()     # accuracy of always predicting "no diabetes"

plt.figure(figsize=(5, 4))
plt.bar(["Always 'no'\n(baseline)", "Logistic\nregression"], [baseline, accuracy], color=["grey", "steelblue"])
for i, v in enumerate([baseline, accuracy]):
    plt.text(i, v + 0.002, f"{v:.3f}", ha="center")
plt.ylim(0.85, 1.0)
plt.title("Accuracy")
plt.show()

**Comment on accuracy:** the model is right **95.9%** of the time. That sounds excellent, but a "model" that always says *no diabetes* would already score **91.2%** (because only ~8.8% of people in the cleaned data have diabetes). So accuracy alone overstates how good the model is. We need the confusion matrix and recall/precision to see how well it finds the people who **do** have diabetes.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["No diabetes", "Diabetes"], cmap="Blues")
plt.title("Confusion matrix")
plt.show()
print(confusion_matrix(y_test, y_pred))

**Comment on the confusion matrix** (19,226 test people):
- **True negatives: 17,354.** Healthy people correctly identified as healthy.
- **False positives: 176.** Healthy people wrongly flagged as diabetic (few, which is good).
- **False negatives: 614.** People **with diabetes that the model missed**. This is the most important error in a medical setting.
- **True positives: 1,082.** People with diabetes correctly identified.

The model is very good at recognising healthy people, but it misses more than a third of the diabetic cases.

In [ ]:
scores = {"Precision": precision_score(y_test, y_pred),
          "Recall": recall_score(y_test, y_pred),
          "F1-score": f1_score(y_test, y_pred)}

plt.figure(figsize=(6, 4))
plt.bar(scores.keys(), scores.values(), color=["#4c72b0", "#dd8452", "#55a868"])
for i, v in enumerate(scores.values()):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.ylim(0, 1.05)
plt.title("Precision, Recall and F1-score (positive class = diabetes)")
plt.show()

**Comment on precision, recall and F1:**
- **Precision = 0.86:** when the model says "diabetes", it is right 86% of the time.
- **Recall = 0.64:** the model only catches **64% of the people who really have diabetes**; the other 36% are missed.
- **F1-score = 0.73:** the balance of the two, pulled down by the lower recall.

For a screening tool, missing a sick person is worse than a false alarm, so we would want **higher recall**. We could get it by lowering the decision threshold (e.g. flagging anyone with probability above 0.3 instead of 0.5) or by training with `class_weight="balanced"`, accepting more false positives in exchange.

## 🌟 Exercise 5: Visualizing the Performance of our Model
A decision boundary can only be drawn in 2D, so we train a logistic regression on the **two strongest features**, `HbA1c_level` and `blood_glucose_level`, and plot where it separates the two classes.

In [ ]:
features_2d = ["HbA1c_level", "blood_glucose_level"]
scaler_2d = StandardScaler()
X2_train = scaler_2d.fit_transform(X_train[features_2d])
X2_test = scaler_2d.transform(X_test[features_2d])

model_2d = LogisticRegression(max_iter=1000).fit(X2_train, y_train)
acc_2d = accuracy_score(y_test, model_2d.predict(X2_test))

# Grid of points covering the plot, coloured by the model's prediction
x_min, x_max = X2_test[:, 0].min() - 0.5, X2_test[:, 0].max() + 0.5
y_min, y_max = X2_test[:, 1].min() - 0.5, X2_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
zz = model_2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

rng = np.random.default_rng(42)
sample = rng.choice(len(X2_test), size=3000, replace=False)   # plot a sample so points stay visible

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, zz, levels=[0, 0.5, 1], colors=["#cfe2f3", "#f4cccc"], alpha=0.8)
plt.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=1.5)
plt.scatter(X2_test[sample, 0], X2_test[sample, 1], c=y_test.to_numpy()[sample], cmap="coolwarm", s=8, alpha=0.6)
plt.xlabel("HbA1c level (standardized)")
plt.ylabel("Blood glucose level (standardized)")
plt.title(f"Decision boundary (2 features) – test accuracy {acc_2d:.3f}")
plt.show()

**Comment:** with just these two blood measurements, the model already reaches **95.5%** accuracy, almost the same as with all features (95.9%). This confirms that **HbA1c and blood glucose** carry most of the information, as their large coefficients suggested. The boundary is a **straight line** (logistic regression is a linear model): people with high values on both measurements fall into the "diabetes" region (red). The points near the line, where the two classes overlap, are the ones the model gets wrong.

## 🌟 Exercise 6: ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"Logistic regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess (AUC = 0.5)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

**Comment on the ROC curve:** the curve rises steeply toward the top-left corner, and the **AUC is 0.96**, far above 0.5 (random guessing). This means the model ranks people very well by risk: a randomly chosen diabetic person gets a higher predicted probability than a randomly chosen healthy person **96%** of the time. Each point on the curve is a different threshold, so the curve shows we can move to a lower threshold and gain a lot of recall while only adding a modest number of false positives.